In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformer, util
import torch

print(torch.cuda.is_available())
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# 1. Load the model
# SentenceTransformer detects device automatically literaly we don't need to add device=device
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

c:\Users\COM\miniconda3\envs\hf-llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Load the Touche-2020 IR dataset (https://huggingface.co/datasets/mteb/webis-touche2020-v3)
corpus = load_dataset("mteb/webis-touche2020-v3", "corpus", split="corpus")
queries = load_dataset("mteb/webis-touche2020-v3", "queries", split="train")
relevant_docs_data = load_dataset("mteb/webis-touche2020-v3", "default", split="test")

In [3]:
# 3. Convert the 'text' column of your corpus into embeddings
# Note: Use convert_to_tensor=True to speed up the search with GPU
corpus_embeddings = model.encode(corpus['text'], convert_to_tensor=True, show_progress_bar=True)

Batches: 100%|██████████| 9492/9492 [05:18<00:00, 29.80it/s] 


In [4]:
# 3. Define your search query
query = "What are the arguments for nuclear energy?"

# 4. Encode the query
query_embedding = model.encode(query, convert_to_tensor=True)

# 5. Search for the top 5 most similar documents
search_results = util.semantic_search(query_embedding, corpus_embeddings, top_k=5)

# 6. Display the results
for hit in search_results[0]:
    doc_id = hit['corpus_id']
    score = hit['score']
    print(f"Score: {score:.4f}")
    print(f"Document: {corpus[doc_id]['text'][:200]}...") # Print first 200 characters
    print("-" * 50)


Score: 0.7036
Document: Although nuclear power seems like the ideal source of energy, there are several problems, each severe and inherent to nuclear power. This video(1) is a pretty good source of information. In essence, t...
--------------------------------------------------
Score: 0.6890
Document: Need both sides of this argument. If you are for nuclearization, why? If you are against nuclearisation, why? Would appreciate it if they are unique opinions - plz don't just say that they are "good" ...
--------------------------------------------------
Score: 0.6877
Document: I'm in favour for nuclear energy.For economical reasons, nuclear energy has strong benefits. Like my opponent said, nuclear power plants are expensive to build and we need to use energy for building i...
--------------------------------------------------
Score: 0.6698
Document: Nuclear energy is a good source of energy, if not the best. It creates jobs, more power, than many types of energy, and has some of te lowe